In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

# xgboost is NOT a component of Python, nor can import.  Must be INSTALL on local and import after that
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
# -----------------------------
# Load Data, and see what it is look like first
# -----------------------------
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')


print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (4209, 378)
Test shape: (4209, 377)


In [ ]:
# -----------------------------
# Identify & Remove Zero-Variance Columns
# -----------------------------
# Compute variance for all numeric columns

    # From the DataFrame train_df, select only the columns whose data type is numeric (like integers or floats), then take just the names of those columns, and assign those names to the variable numeric_cols
    # select_dtypes is a pandas DataFrame method that filters columns based on their data type (e.g., int64, float64, object, datetime).  It returns a new DataFrame containing only the columns whose data types match the criteria you specify.
    # It returns a new DataFrame containing only the columns whose data types match the criteria you specify.
numeric_cols = train_df.select_dtypes(include=[np.number]).columns

    # any columns with variance == 0, store it in the variable zero_var_cols
zero_var_cols = train_df[numeric_cols].var() == 0

    # zero_var_cols[zero_var_cols] check if the TRUE columns exist -> grab the name -> put to a list and assign to variable cols_to_drop
cols_to_drop = zero_var_cols[zero_var_cols].index.tolist()


In [4]:
# Also check non-numeric (e.g., constant strings)
constant_non_numeric = []
for col in train_df.columns:
    if train_df[col].nunique() == 1:
        constant_non_numeric.append(col)

cols_to_drop += constant_non_numeric
cols_to_drop = list(set(cols_to_drop))  # dedupe

train_df.drop(columns=cols_to_drop, inplace=True)
test_df.drop(columns=[c for c in cols_to_drop if c in test_df.columns], inplace=True)


In [5]:
# -----------------------------
# Check Nulls & Unique Values
# -----------------------------
print("\n--- NULL CHECK ---")
print("Train nulls:", train_df.isnull().sum().sum())
print("Test nulls:", test_df.isnull().sum().sum())

print("\n--- UNIQUE VALUE EXAMPLES (first 5 cols) ---")
for col in train_df.columns[:5]:
    print(f"{col}: {train_df[col].nunique()} unique values")



--- NULL CHECK ---
Train nulls: 0
Test nulls: 0

--- UNIQUE VALUE EXAMPLES (first 5 cols) ---
ID: 4209 unique values
y: 2545 unique values
X0: 47 unique values
X1: 27 unique values
X2: 44 unique values


In [6]:
# -----------------------------
# Separate Target & Features
# -----------------------------
y_train = train_df['y']  # assuming target column is named 'y'
X_train = train_df.drop(columns=['y'])
X_test = test_df.copy()


In [7]:
# -----------------------------
# Label Encoding for Categorical Columns
# -----------------------------
# Identify object-type columns (assumed to be categorical)
cat_cols = X_train.select_dtypes(include=['object']).columns

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    # Fit on union of train + test to avoid unseen label error
    all_vals = pd.concat([X_train[col], X_test[col]], axis=0).astype(str)
    le.fit(all_vals)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

print(f"\nEncoded {len(cat_cols)} categorical columns.")


Encoded 8 categorical columns.


C:\Users\SETUP\AppData\Local\Temp\ipykernel_34712\627469867.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns


In [9]:
# -----------------------------
# Dimensionality Reduction (PCA)
# -----------------------------
# Optional: Only apply if too many features (>1000)
if X_train.shape[1] > 500:
    print("Applying PCA...")
    pca = PCA(n_components=0.95)  # retain 95% variance
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)
    print(f"Reduced from {X_train.shape[1]} to {X_train_pca.shape[1]} features.")
else:
    print("Skipping PCA (feature count <= 500)")
    X_train_pca = X_train.values
    X_test_pca = X_test.values

Skipping PCA (feature count <= 500)


In [10]:
# -----------------------------
# 7. Train XGBoost & Predict
# -----------------------------
print("\nTraining XGBoost...")
model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='rmse'
)

model.fit(X_train_pca, y_train)

# Predict on test set
y_pred = model.predict(X_test_pca)


Training XGBoost...


In [11]:
# -----------------------------
# 8. Save Submission
# -----------------------------
submission = pd.DataFrame({'ID': test_df.index, 'y': y_pred})
submission.to_csv('submission.csv', index=False)
print("\n✅ Prediction saved to 'submission.csv'")


✅ Prediction saved to 'submission.csv'
